In [5]:
import pandas as pd

# ---------------------------
# 1. Load Dataset
# ---------------------------
df = pd.read_csv("/content/sample_data/cardio_train.csv", sep=';')

# ---------------------------
# 2. Preprocessing
# ---------------------------
df["age_years"] = (df["age"] / 365).astype(int)
df["BMI"] = df["weight"] / ((df["height"] / 100) ** 2)

# ---------------------------
# 3. Model-1 Interpretation (Reuse)
# ---------------------------

def interpret_bp_sys(value):
    if value < 90:
        return "Low"
    elif value > 120:
        return "High"
    else:
        return "Normal"

def interpret_bp_dia(value):
    if value < 60:
        return "Low"
    elif value > 80:
        return "High"
    else:
        return "Normal"

df["Systolic_BP_Status"] = df["ap_hi"].apply(interpret_bp_sys)
df["Diastolic_BP_Status"] = df["ap_lo"].apply(interpret_bp_dia)

df["Glucose_Status"] = df["gluc"].apply(lambda x: "Normal" if x == 1 else "High")
df["Cholesterol_Status"] = df["cholesterol"].apply(lambda x: "Normal" if x == 1 else "High")

df["BMI_Status"] = df["BMI"].apply(
    lambda x: "Normal" if 18.5 <= x <= 24.9 else ("High" if x >= 30 else "Overweight")
)

# ---------------------------
# 4. Model-2: Pattern Recognition
# ---------------------------

def detect_hypertension(row):
    if row["Systolic_BP_Status"] == "High" or row["Diastolic_BP_Status"] == "High":
        return "Yes"
    return "No"

def detect_obesity(bmi):
    return "Yes" if bmi >= 30 else "No"

def metabolic_risk(row):
    if row["Glucose_Status"] == "High" and row["Cholesterol_Status"] == "High":
        return "Yes"
    return "No"

df["Hypertension_Risk"] = df.apply(detect_hypertension, axis=1)
df["Obesity_Risk"] = df["BMI"].apply(detect_obesity)
df["Metabolic_Risk"] = df.apply(metabolic_risk, axis=1)

# ---------------------------
# 5. Cardio Risk Score
# ---------------------------

def cardio_risk_score(row):
    score = 0
    if row["Hypertension_Risk"] == "Yes":
        score += 2
    if row["Obesity_Risk"] == "Yes":
        score += 2
    if row["Metabolic_Risk"] == "Yes":
        score += 2
    if row["age_years"] > 50:
        score += 1
    if row["smoke"] == 1:
        score += 1
    return score

df["Cardio_Risk_Score"] = df.apply(cardio_risk_score, axis=1)

# ---------------------------
# 6. Risk Level Classification
# ---------------------------

def risk_level(score):
    if score <= 2:
        return "Low"
    elif score <= 5:
        return "Medium"
    else:
        return "High"

df["Overall_Risk_Level"] = df["Cardio_Risk_Score"].apply(risk_level)

# ---------------------------
# 7. Final Output (Week-2)
# ---------------------------
final_columns = [
    "age_years",
    "Glucose_Status",
    "Cholesterol_Status",
    "Systolic_BP_Status",
    "Diastolic_BP_Status",
    "BMI_Status",
    "Hypertension_Risk",
    "Obesity_Risk",
    "Metabolic_Risk",
    "Cardio_Risk_Score",
    "Overall_Risk_Level"
]

print("\nWEEK-2 PATTERN RECOGNITION OUTPUT\n")
print(df[final_columns].head())

# ---------------------------
# 8. Save Output
# ---------------------------
df.to_csv("week2_cardio_pattern_analysis.csv", index=False)

print("\n✅ Week-2 Task Completed Successfully")


WEEK-2 PATTERN RECOGNITION OUTPUT

   age_years Glucose_Status Cholesterol_Status Systolic_BP_Status  \
0         50         Normal             Normal             Normal   
1         55         Normal               High               High   
2         51         Normal               High               High   
3         48         Normal             Normal               High   
4         47         Normal             Normal             Normal   

  Diastolic_BP_Status  BMI_Status Hypertension_Risk Obesity_Risk  \
0              Normal      Normal                No           No   
1                High        High               Yes          Yes   
2              Normal      Normal               Yes           No   
3                High  Overweight               Yes           No   
4              Normal      Normal                No           No   

  Metabolic_Risk  Cardio_Risk_Score Overall_Risk_Level  
0             No                  0                Low  
1             No          